In [1]:
import os
import socket
from dotenv import load_dotenv
from phi.agent import Agent  
from phi.model.groq import Groq  
from phi.tools.duckduckgo import DuckDuckGo  
from phi.tools.yfinance import YFinanceTools 

### GROQ
* Answers are generated using [GROQ free api](https://console.groq.com/home)

In [2]:
# Loading environment variables
if load_dotenv():
    print("[INFO] Env variables sucessfully loaded.")
else:
    print("[WARNING] .env file not found. Some settings may be missing.")


[INFO] Env variables sucessfully loaded.


In [5]:
# Check internet connection
def check_connection():
    try:
        socket.create_connection(("www.google.com", 80), timeout=5)
        print("[INFO] Internet connection checked.")
        return True
    except OSError:
        print("[ERROR] No internet connection! Some frameworks may not work.")
        return False

In [6]:
flag_conn = check_connection()

# Creating AI Agents
print("[INFO] Starting agents...")

[INFO] Internet connection checked.
[INFO] Starting agents...


In [12]:
# Financial Agent
fin_agent = Agent(name="Financial Agent",  
                  model=Groq(id="openai/gpt-oss-120b"),
                  tools=[YFinanceTools(stock_price=True,  
                                       analyst_recommendations=True,  
                                       stock_fundamentals=True)] if flag_conn else [],
                                       show_tool_calls=True,  
                                       markdown=True,  
                                       instructions=["Generate tables to comparasions", "Generate the results in English"])

In [13]:
# Search Agent
search_agent = Agent(name="Web Searcher",  
                            model=Groq(id="openai/gpt-oss-120b"),
                            tools=[DuckDuckGo()] if flag_conn else [],
                            show_tool_calls=True,  
                            markdown=True,  
                            instructions=["You always need to include the references you collect",
                                          "Generate the results in English",
                                          "Use always reliable sources, you are a senior financial analyst"
                                          ])

In [14]:
# Team agents
team_agents = Agent(team=[fin_agent, search_agent], 
                         model=Groq(id="openai/gpt-oss-120b"),
                         show_tool_calls=True,  
                         markdown=True,  
                         instructions=["Includes the sources of the collected informations always",
                                       "Create tables to compare",
                                       "Generate the results in English"
                                       ],
                         debug_mode=False)

In [15]:
response = team_agents.run(
    "Summarize the recommendations from analysts about Netflix investments and share the latest information and news."
)

print(response)

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\phi\tools\duckduckgo.py:86: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS(
c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\phi\tools\duckduckgo.py:86: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS(


WARNING  Could not run function duckduckgo_news(max_results=10, query=Netflix September 2026 earnings)

ERROR    https://duckduckgo.com/news.js?l=us-en&o=json&noamp=1&q=Netflix+September+2026+earnings&vqd=4-945790181491
         28009793493499601024875918&p=-1 403 Ratelimit                                                             
         Traceback (most recent call last):                                                                        
           File "c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\phi\tools\function.py", line 368, in      
         execute                                                                                                   
             self.result = self.function.entrypoint(**entrypoint_args, **self.arguments)                           
                           ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                           
           File "c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\pydantic\_internal\_validate_call.py",    
         line 40, in wrapper_function                                                                              
             return wrapper(*args, **kwargs)                                                                       
           File "c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\pydantic\_internal\_validate_call.py",    
         line 137, in __call__                                                                                     
             res = self.__pydantic_validator__.validate_python(pydantic_core.ArgsKwargs(args, kwargs))             
           File "c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\phi\tools\duckduckgo.py", line 89, in     
         duckduckgo_news                                                                                           
             return json.dumps(ddgs.news(keywords=query, max_results=(self.fixed_max_results or max_results)),     
         indent=2)                                                                                                 
                               ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^      
           File "c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\duckduckgo_search\duckduckgo_search.py",  
         line 646, in news                                                                                         
             resp_content = self._get_url("GET", "https://duckduckgo.com/news.js", params=payload).content         
                            ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                 
           File "c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\duckduckgo_search\duckduckgo_search.py",  
         line 138, in _get_url                                                                                     
             raise RatelimitException(f"{resp.url} {resp.status_code} Ratelimit")                                  
         duckduckgo_search.exceptions.RatelimitException:                                                          
         https://duckduckgo.com/news.js?l=us-en&o=json&noamp=1&q=Netflix+September+2026+earnings&vqd=4-945790181491
         28009793493499601024875918&p=-1 403 Ratelimit

c:\Users\krupc\anaconda3\envs\pyagent\Lib\site-packages\phi\tools\duckduckgo.py:65: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddgs = DDGS(


KeyboardInterrupt: 

In [ ]:
# Checking if there is valid content before saving
if response.content.strip():
    with open("results.md", "w", encoding="utf-8") as arquivo:
        arquivo.write("# ANALYST FINANCIAL REPORT\n\n")
        arquivo.write(response.content)

    print("[INFO] The results were saved in results.md")
else:
    print("[ERROR] The answer of the agent is empty.")

In [ ]:
print(response.content)